# NexuStock · Construcción del dataset semanal

| | |
|---|---|
| **Proyecto** | NexuStock: gestión y predicción de inventario para una carnicería independiente |
| **Entregable** | Dataset procesado · Kanban v3, tarea 25 |
| **Responsable** | Manuel Bustamante (Data Scientist) |
| **Entrada** | `data/raw/`, `data/processed/stock_diario.csv` (notebook 01) |
| **Salida** | `data/processed/semanal.csv` |

## Objetivo

Construir una tabla con una fila por producto y semana, donde `kg` es la variable objetivo y el resto de las columnas contiene solo información disponible antes del inicio de esa semana. Las variables responden a los hallazgos del notebook 01 y a los requisitos del ERS.

## Criterios de construcción

- La semana va de lunes a domingo y se identifica por la fecha del lunes.
- Se excluyen las semanas incompletas en los extremos del periodo.
- Las variables se calculan sobre la serie completa 2022-2024, porque los lags de enero de 2024 requieren las semanas de diciembre de 2023. La separación entre entrenamiento y prueba se aplica al final, por fecha.
- Ninguna variable usa información de la semana que se predice, salvo el calendario, que se conoce de antemano.

## 1. Configuración y fechas de corte

El 31 de diciembre de 2023 es domingo, por lo que el corte entre entrenamiento y prueba coincide con el cierre de una semana y ninguna semana queda repartida entre ambos conjuntos.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:,.2f}'.format)
sns.set_theme(style='whitegrid')

BASE = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW = BASE / 'data' / 'raw'
PROCESSED = BASE / 'data' / 'processed'

FIN_ENTRENAMIENTO = pd.Timestamp('2023-12-31')
PRIMERA_SEMANA = pd.Timestamp('2022-01-03')   # primer lunes con semana completa
ULTIMA_SEMANA = pd.Timestamp('2024-12-23')    # último lunes con semana completa


## 2. Carga de datos

Cada fecha se asigna al lunes de su semana.

In [2]:
ventas = pd.concat([pd.read_csv(f) for f in sorted(RAW.glob('20*/ventas_*.csv'))], ignore_index=True)
ventas['fecha'] = pd.to_datetime(ventas['fecha'])
productos = pd.read_csv(RAW / 'productos.csv')
calendario = pd.read_csv(RAW / 'calendario_eventos.csv', parse_dates=['fecha_inicio', 'fecha_fin'])
stock = pd.read_csv(PROCESSED / 'stock_diario.csv', parse_dates=['fecha'])

# Lunes de la semana a la que pertenece cada fecha
ventas['semana'] = ventas['fecha'] - pd.to_timedelta(ventas['fecha'].dt.dayofweek, unit='D')
stock['semana'] = stock['fecha'] - pd.to_timedelta(stock['fecha'].dt.dayofweek, unit='D')
print(ventas.shape, stock.shape, calendario.shape)


(106224, 11) (26304, 8) (99, 6)


## 3. Grilla producto × semana

Se genera una fila por cada combinación de producto y semana, incluidas las semanas sin venta, que quedan con 0 kg. Sin esta grilla, las semanas sin venta desaparecerían y los lags se desalinearían: `shift(1)` dejaría de corresponder a la semana anterior.

In [3]:
semanas = pd.date_range(PRIMERA_SEMANA, ULTIMA_SEMANA, freq='W-MON')
grilla = pd.MultiIndex.from_product([productos['producto_id'], semanas], names=['producto_id', 'semana'])

kg = ventas.groupby(['producto_id', 'semana'])['cantidad_kg'].sum()
df = kg.reindex(grilla, fill_value=0).rename('kg').reset_index()
df = df.merge(productos[['producto_id', 'categoria']], on='producto_id', how='left')

print('Semanas:', len(semanas), '| Filas:', len(df), '| Semanas-producto sin venta:', (df['kg'] == 0).sum())
df.head()


Semanas: 156 | Filas: 3744 | Semanas-producto sin venta: 6


,producto_id,semana,kg,categoria
0,P001,2022-01-03,40.12,Vacuno
1,P001,2022-01-10,41.50,Vacuno
2,P001,2022-01-17,30.21,Vacuno
3,P001,2022-01-24,33.58,Vacuno
4,P001,2022-01-31,36.54,Vacuno


**Resultado.** 156 semanas × 24 productos = 3.744 filas. Solo 6 combinaciones producto-semana no registran venta.

## 4. Variables de calendario

Cada día se marca según los eventos del calendario y luego se resume por semana. Los eventos se conocen con anticipación, por lo que pueden usarse para la misma semana que se predice.

| Variable | Definición | Resumen semanal |
|---|---|---|
| `dias_cerrado` | Días con el local cerrado (feriado irrenunciable o elección) | Suma |
| `feriado_largo` | La semana contiene un fin de semana largo | Máximo (0/1) |
| `fiestas_patrias` | La semana cae en el periodo de Fiestas Patrias | Máximo (0/1) |
| `semana_santa` | La semana contiene Semana Santa, Viernes Santo o Sábado Santo | Máximo (0/1) |
| `cuaresma` | La semana cae en Cuaresma | Máximo (0/1) |
| `evento_especial` | Navidad, Año Nuevo o Día del Padre | Máximo (0/1) |
| `quincena` | La semana contiene el día 15 | Máximo (0/1) |
| `fin_mes` | La semana contiene el último día del mes | Máximo (0/1) |

`semana_santa` se filtra por nombre del evento y no por tipo, porque el calendario clasifica el Día de Todos los Santos (1 de noviembre) con el tipo `semana_santa`.

In [4]:
dias = pd.DataFrame({'fecha': pd.date_range(PRIMERA_SEMANA, ULTIMA_SEMANA + pd.Timedelta(days=6), freq='D')})
dias['semana'] = dias['fecha'] - pd.to_timedelta(dias['fecha'].dt.dayofweek, unit='D')

def marcar(dias, filtro):
    """True en los días cubiertos por algún evento del calendario que cumpla el filtro."""
    marca = pd.Series(False, index=dias.index)
    for _, ev in calendario[filtro].iterrows():
        marca |= dias['fecha'].between(ev['fecha_inicio'], ev['fecha_fin'])
    return marca

ev = calendario['evento']
dias['cerrado'] = marcar(dias, calendario['efecto_esperado'].str.contains('Local cerrado'))
dias['feriado_largo'] = marcar(dias, calendario['tipo'] == 'feriado_largo')
dias['fiestas_patrias'] = marcar(dias, ev.str.startswith('Fiestas Patrias'))
dias['semana_santa'] = marcar(dias, ev.str.contains('Semana Santa|Viernes Santo|Sábado Santo'))
dias['cuaresma'] = marcar(dias, ev.str.startswith('Cuaresma'))
dias['evento_especial'] = marcar(dias, (calendario['tipo'] == 'otro') & ~ev.str.startswith('Cierre Paso'))
dias['quincena'] = dias['fecha'].dt.day == 15
dias['fin_mes'] = dias['fecha'].dt.is_month_end

cal_semana = dias.groupby('semana').agg(
    dias_cerrado=('cerrado', 'sum'),
    feriado_largo=('feriado_largo', 'max'),
    fiestas_patrias=('fiestas_patrias', 'max'),
    semana_santa=('semana_santa', 'max'),
    cuaresma=('cuaresma', 'max'),
    evento_especial=('evento_especial', 'max'),
    quincena=('quincena', 'max'),
    fin_mes=('fin_mes', 'max'),
).astype(int)
cal_semana.sum()


dias_cerrado       19
feriado_largo      19
fiestas_patrias     8
semana_santa        6
cuaresma           18
evento_especial     9
quincena           36
fin_mes            35
dtype: int64

## 5. Días hasta el 18, estacionalidad y tendencia

- `dias_hasta_18`: días entre el lunes de la semana y el próximo 18 de septiembre.
- `semana_anio` y `mes`: posición en el año. El mes se toma del jueves, de modo que la semana se asigna al mes que contiene la mayoría de sus días (criterio ISO 8601).
- `t`: índice correlativo de semana, que representa la tendencia.

In [5]:
def dias_hasta_18(lunes):
    dieciocho = pd.Timestamp(lunes.year, 9, 18)
    if lunes > dieciocho:
        dieciocho = pd.Timestamp(lunes.year + 1, 9, 18)
    return (dieciocho - lunes).days

cal_semana['dias_hasta_18'] = [dias_hasta_18(s) for s in cal_semana.index]
cal_semana['semana_anio'] = cal_semana.index.isocalendar().week.astype(int)
cal_semana['mes'] = (cal_semana.index + pd.Timedelta(days=3)).month   # mes del jueves de la semana
cal_semana['t'] = np.arange(len(cal_semana))                          # índice de tendencia
cal_semana.loc['2023-08-28':'2023-09-25']


,dias_cerrado,feriado_largo,fiestas_patrias,semana_santa,cuaresma,evento_especial,quincena,fin_mes,dias_hasta_18,semana_anio,mes,t
semana,,,,,,,,,,,,
2023-08-28,0,0,0,0,0,0,0,1,21,35,8,86
2023-09-04,0,0,1,0,0,0,0,0,14,36,9,87
2023-09-11,0,1,1,0,0,0,1,0,7,37,9,88
2023-09-18,2,0,1,0,0,0,0,0,0,38,9,89
2023-09-25,0,0,0,0,0,0,0,1,359,39,9,90


**Resultado.** La cuenta regresiva baja de 21 a 0 en las semanas previas al 18 y vuelve a 359 en la semana siguiente. La semana del 18 de septiembre de 2023 registra 2 días cerrados. La discontinuidad de `dias_hasta_18` limita su utilidad en un modelo lineal, mientras que un modelo de árboles puede aislar el tramo de pocos días mediante cortes por umbral.

## 6. Integración del calendario y los quiebres

In [6]:
df = df.merge(cal_semana.reset_index(), on='semana', how='left')

quiebres = stock.groupby(['producto_id', 'semana'])['quiebre'].sum().rename('dias_quiebre').reset_index()
df = df.merge(quiebres, on=['producto_id', 'semana'], how='left')
print('Nulos:', df.isna().sum().sum(), '| Forma:', df.shape)


Nulos: 0 | Forma: (3744, 17)


## 7. Lags y medias móviles

Ventas de las 1 a 4 semanas anteriores y promedios de las últimas 4 y 8 semanas, calculados por producto. El desplazamiento (`shift(1)`) se aplica antes del promedio móvil, de modo que `media_4` y `media_8` excluyen la semana que se predice.

In [7]:
df = df.sort_values(['producto_id', 'semana']).reset_index(drop=True)
por_producto = df.groupby('producto_id')

for k in (1, 2, 3, 4):
    df[f'lag_{k}'] = por_producto['kg'].shift(k)
df['media_4'] = por_producto['kg'].transform(lambda s: s.shift(1).rolling(4).mean())
df['media_8'] = por_producto['kg'].transform(lambda s: s.shift(1).rolling(8).mean())
df['dias_quiebre_lag_1'] = por_producto['dias_quiebre'].shift(1)

df.loc[df['producto_id'] == 'P001', ['semana', 'kg', 'lag_1', 'lag_2', 'media_4']].head(6)


,semana,kg,lag_1,lag_2,media_4
0,2022-01-03,40.12,NaN,NaN,NaN
1,2022-01-10,41.50,40.12,NaN,NaN
2,2022-01-17,30.21,41.50,40.12,NaN
3,2022-01-24,33.58,30.21,41.50,NaN
4,2022-01-31,36.54,33.58,30.21,36.35
5,2022-02-07,38.84,36.54,33.58,35.46


**Verificación manual.** Para la semana del 31-01-2022, `media_4` = (40,12 + 41,50 + 30,21 + 33,58) / 4 = 36,35, sin incluir los 36,54 kg de esa semana.

## 8. Prueba de ausencia de fuga de información

Comprobación automática de que `lag_1` es exactamente la venta de la semana anterior y de que `media_4` usa las cuatro semanas previas. La celda se mantiene en el notebook como evidencia de que no hay fuga de información.

In [8]:
p = df[df['producto_id'] == 'P001'].reset_index(drop=True)
assert (p['lag_1'].iloc[1:].values == p['kg'].iloc[:-1].values).all()
assert np.isclose(p.loc[10, 'media_4'], p.loc[6:9, 'kg'].mean())
print('Lags correctos: solo usan semanas anteriores')


Lags correctos: solo usan semanas anteriores


## 9. Limpieza y separación de conjuntos

Se eliminan las primeras 8 semanas de cada producto, que no tienen historial suficiente para calcular `media_8` (8 × 24 = 192 filas). Se agregan dos columnas que no son variables del modelo:

- `conjunto`: entrenamiento hasta el 31-12-2023, prueba desde el 01-01-2024.
- `es_festiva`: marca las ventanas festivas definidas en el ERS (Semana Santa, Fiestas Patrias y feriados largos) para calcular el MAE segmentado.

In [9]:
antes = len(df)
df = df.dropna(subset=['media_8']).reset_index(drop=True)
df['conjunto'] = np.where(df['semana'] <= FIN_ENTRENAMIENTO, 'entrenamiento', 'prueba')
df['es_festiva'] = ((df['semana_santa'] == 1) | (df['fiestas_patrias'] == 1) | (df['feriado_largo'] == 1)).astype(int)

print('Filas eliminadas por falta de historial:', antes - len(df))
df.groupby('conjunto').agg(filas=('kg', 'size'), semanas=('semana', 'nunique'),
                           desde=('semana', 'min'), hasta=('semana', 'max'),
                           festivas=('es_festiva', 'sum'))


Filas eliminadas por falta de historial: 192


,filas,semanas,desde,hasta,festivas
conjunto,,,,,
entrenamiento,2304,96,2022-02-28,2023-12-25,528
prueba,1248,52,2024-01-01,2024-12-23,120


**Resultado.**

| Conjunto | Filas | Semanas | Periodo | Filas festivas |
|---|---|---|---|---|
| Entrenamiento | 2.304 | 96 | 2022-02-28 a 2023-12-25 | 528 |
| Prueba | 1.248 | 52 | 2024-01-01 a 2024-12-23 | 120 |

## 10. Relación de las variables con la desviación de la venta

Se calcula la correlación de cada variable con `ratio` = `kg` / `media_8`, es decir, cuánto se aparta la venta de una semana de su nivel reciente. Una correlación directa con `kg` estaría dominada por las diferencias de volumen entre productos. El análisis usa solo el conjunto de entrenamiento.

In [10]:
ent = df[df['conjunto'] == 'entrenamiento'].copy()
ent['ratio'] = ent['kg'] / ent['media_8']

corr = ent.select_dtypes('number').drop(columns=['kg', 'dias_quiebre']).corr()['ratio'].drop('ratio')
corr.sort_values(key=abs, ascending=False).round(2)


feriado_largo         0.17
fiestas_patrias       0.15
dias_cerrado         -0.12
es_festiva            0.11
dias_hasta_18        -0.10
quincena              0.10
fin_mes               0.07
media_8              -0.06
lag_2                -0.05
mes                   0.04
lag_4                -0.04
media_4              -0.04
semana_anio           0.04
lag_3                -0.04
semana_santa         -0.03
lag_1                -0.03
t                     0.02
evento_especial       0.02
dias_quiebre_lag_1    0.01
cuaresma              0.01
Name: ratio, dtype: float64

**Resultado.** Las variables con mayor relación son `feriado_largo` (0,17), `fiestas_patrias` (0,15), `dias_cerrado` (−0,12), `dias_hasta_18` (−0,10), `quincena` (0,10) y `fin_mes` (0,07). `semana_santa` muestra una correlación cercana a 0 (−0,03) porque eleva la venta de pescados y reduce la de carnes rojas, y ambos efectos se compensan en el agregado.

## 11. Efecto de los eventos en el conjunto de entrenamiento

In [11]:
display(ent.groupby('fiestas_patrias')['ratio'].mean().round(2))
display(ent.groupby('dias_cerrado')['ratio'].mean().round(2))
ent[ent['categoria'] == 'Pescados'].groupby('semana_santa')['kg'].mean().round(2)


fiestas_patrias
0   1.00
1   1.19
Name: ratio, dtype: float64

dias_cerrado
0   1.02
1   1.04
2   0.61
Name: ratio, dtype: float64

semana_santa
0   14.59
1   24.30
Name: kg, dtype: float64

**Resultado.**

- Las semanas de Fiestas Patrias venden 19 % sobre su nivel reciente (ratio 1,19).
- Con 2 días cerrados la venta cae a 61 % de su nivel reciente; con 1 día no hay efecto apreciable (1,04).
- Los pescados promedian 24,30 kg en Semana Santa frente a 14,59 kg en el resto de las semanas.

## 12. Exportación del dataset

In [12]:
COLUMNAS = ['producto_id', 'categoria', 'semana', 'conjunto', 'kg',
            'lag_1', 'lag_2', 'lag_3', 'lag_4', 'media_4', 'media_8',
            'dias_hasta_18', 'fiestas_patrias', 'semana_santa', 'cuaresma', 'feriado_largo',
            'evento_especial', 'quincena', 'fin_mes', 'dias_cerrado',
            'semana_anio', 'mes', 't', 'dias_quiebre_lag_1', 'dias_quiebre', 'es_festiva']
df[COLUMNAS].to_csv(PROCESSED / 'semanal.csv', index=False)
print('Guardado:', PROCESSED / 'semanal.csv', df[COLUMNAS].shape)


Guardado: c:\Users\manu_\OneDrive\Escritorio\NexuStock\NexuStock\Codigo\ml-engine\data\processed\semanal.csv (3552, 26)


## Diccionario de datos · `semanal.csv`

| Columna | Tipo | Descripción | Uso |
|---|---|---|---|
| `producto_id`, `semana` | Clave | Producto y lunes de la semana | Identificación |
| `categoria` | Categórica | Vacuno, Cerdo, Pollo, Embutidos o Pescados | Variable |
| `kg` | Numérica | kg vendidos en la semana | **Objetivo** |
| `lag_1` a `lag_4` | Numérica | kg vendidos 1 a 4 semanas antes | Variable |
| `media_4`, `media_8` | Numérica | Promedio de las 4 y 8 semanas anteriores | Variable; `media_8` es también la línea base |
| `dias_hasta_18` | Numérica | Días hasta el próximo 18 de septiembre | Variable |
| `fiestas_patrias`, `semana_santa`, `cuaresma`, `feriado_largo`, `evento_especial` | Binaria | Eventos del calendario en la semana | Variable |
| `quincena`, `fin_mes` | Binaria | La semana contiene un día de pago | Variable |
| `dias_cerrado` | Numérica | Días con el local cerrado | Variable |
| `semana_anio`, `mes`, `t` | Numérica | Estacionalidad y tendencia | Variable |
| `dias_quiebre_lag_1` | Numérica | Días con quiebre en la semana anterior | Variable |
| `dias_quiebre` | Numérica | Días con quiebre en la misma semana | Solo análisis de errores (información futura) |
| `conjunto` | Categórica | Entrenamiento o prueba | Separación |
| `es_festiva` | Binaria | Ventana festiva según el ERS | Solo MAE segmentado |